# Architecture du projet et rôle des fichiers

### `cv/parser.py`
Extraction du texte brut à partir des CV au format PDF afin de rendre leur contenu exploitable par les étapes NLP.

### `cv/preprocess.py`
Nettoyage et normalisation des textes (CV et offres) : suppression du bruit, tokenisation, stopwords et lemmatisation pour préparer la vectorisation.

### `nlp/vectorizer.py`
Construction de la représentation TF-IDF des offres d’emploi et préparation de l’espace vectoriel de référence.

### `nlp/lsa.py`
Réduction de dimension par LSA (SVD) afin de capter des similarités sémantiques globales au-delà du simple vocabulaire commun.

### `nlp/similarity.py`
Calcul des similarités cosinus entre un CV et les offres pour produire un classement des annonces les plus pertinentes.

### `pipeline.py`
Orchestration complète du traitement : chargement des données, entraînement des modèles, calcul des classements TF-IDF et LSA, et export des résultats.

### `main.py`
Point d’entrée du projet permettant d’exécuter l’ensemble du pipeline en une seule commande.

## 1. Extraction des CV — `parser.py`

### Objectif
Ce module est responsable de l’extraction du texte brut à partir des CV au format PDF.  
Il constitue la **première étape du pipeline**, avant tout traitement NLP.

### Fonctionnement
1. Chaque fichier PDF est ouvert page par page à l’aide de la bibliothèque PyMuPDF.
2. Le texte de chaque page est extrait puis stocké.
3. Les textes de toutes les pages sont concaténés pour former un seul document texte par CV.
4. Tous les CV présents dans le dossier `data/cv/` sont chargés automatiquement.
5. Le résultat est un dictionnaire associant le nom du CV à son texte brut.

Ce module ne réalise **aucun nettoyage ni transformation linguistique** :  
il se limite volontairement à la lecture et à l’extraction du contenu.

In [ ]:
# Installation via requirements.txt
pip install -r requirements.txt

In [ ]:
from pathlib import Path
from cv.parser import load_all_cvs

# Dossier contenant les CV
cv_dir = Path("data/cv")

# Chargement de tous les CV
cvs = load_all_cvs(cv_dir)

# Affichage d’un aperçu pour chaque CV
for name, text in cvs.items():
    print(f"\n===== CV : {name} =====")
    print(text[:800])  # aperçu du texte extrait


## 2. Nettoyage & normalisation — `preprocess.py`

### Objectif
Ce module transforme un texte brut (CV ou offre) en un texte **propre, homogène et comparable**.  
L’idée est de réduire le bruit (contacts, ponctuation, caractères spéciaux…) et de garder les mots les plus informatifs.

### Fonctionnement (étapes)
1. **Mise en minuscules** : évite les différences artificielles (ex: "Ingénieur" vs "ingénieur").
2. **Suppression des emails et numéros de téléphone** : informations inutiles pour le matching.
3. **Suppression des caractères non-alphabétiques** (en conservant les lettres accentuées).
4. **Tokenisation simple** : découpage du texte en mots.
5. **Filtrage** :
   - suppression des stopwords français (ex: "le", "et", "avec"),
   - suppression des mots trop courts (≤ 2 caractères).
6. **Lemmatisation** : ramène les mots vers une forme de base pour réduire la variabilité du vocabulaire.
7. **Reconstruction** : les tokens restants sont regroupés en une phrase unique.

### Résultat attendu
Un texte nettoyé prêt à être vectorisé (TF-IDF / LSA), appliqué de la même façon aux CV et aux offres.


In [ ]:
from pathlib import Path
from cv.parser import load_all_cvs
from cv.preprocess import clean_text

cv_dir = Path("data/cv")
cvs = load_all_cvs(cv_dir)

for name, raw_text in cvs.items():
    cleaned = clean_text(raw_text)

    print(f"\n===== CV NETTOYÉ : {name} =====")
    print("---- Aperçu texte brut ----")
    print(raw_text[:400])
    print("\n---- Aperçu texte nettoyé ----")
    print(cleaned[:400])


## 3. Vectorisation TF-IDF des offres — `nlp/vectorizer.py`

### Objectif
Ce module prépare les offres d’emploi pour la comparaison avec les CV en construisant une représentation numérique **TF-IDF**.  
L’idée est de transformer chaque offre en vecteur, où les mots spécifiques ont plus de poids que les mots trop fréquents.

### Fonctionnement (étapes)
1. **Chargement du fichier CSV des offres** (`data/offer.csv`).
2. **Construction d’un “document” texte par offre** en concaténant les colonnes textuelles utiles (ex: titre, description, compétences… selon le dataset).
3. **Nettoyage** de ce document via `clean_text` (même preprocessing que pour les CV).
4. **Entraînement du TF-IDF** sur l’ensemble des offres :
   - création du vocabulaire,
   - calcul des poids TF-IDF,
   - production d’une matrice `X_offers_tfidf` (1 ligne = 1 offre, 1 colonne = 1 terme).
5. Le modèle TF-IDF appris sur les offres sera ensuite utilisé pour transformer un CV dans le même espace (important : pas de ré-apprentissage sur le CV).

### Résultat attendu
- Une matrice TF-IDF des offres (dimension : `nb_offres × nb_termes`)
- Un objet `vectorizer` réutilisable pour transformer n’importe quel CV


In [ ]:
from pathlib import Path
from nlp.vectorizer import load_and_prepare_offers, build_tfidf_matrix

offers_path = Path("data/offer.csv")

# 1) Chargement + préparation des offres (document + nettoyage)
df_offers = load_and_prepare_offers(offers_path)

print("Colonnes disponibles :", list(df_offers.columns))
print("Nombre d'offres :", len(df_offers))

# 2) TF-IDF
vectorizer, X_offers_tfidf = build_tfidf_matrix(df_offers["clean_document"].tolist())

print("\nTF-IDF matrix shape :", X_offers_tfidf.shape)
print("Nombre de termes (vocabulaire) :", len(vectorizer.get_feature_names_out()))

# 3) Exemple : aperçu d'une offre nettoyée
print("\n--- Exemple clean_document (offre 0) ---")
print(df_offers["clean_document"].iloc[0][:400])

## 4. Similarité CV ↔ Offres (TF-IDF) — `nlp/similarity.py`

### Objectif
Ce module calcule la similarité entre un CV et toutes les offres, puis retourne les offres les plus pertinentes selon **TF-IDF + similarité cosinus**.

### Principe
- Les offres sont déjà vectorisées en TF-IDF (matrice `X_offers_tfidf`).
- Le CV est nettoyé avec le même preprocessing, puis transformé avec le **même vectorizer TF-IDF**.
- On calcule une **similarité cosinus** entre le vecteur du CV et chaque vecteur d’offre.
- On trie les offres par score décroissant et on affiche un Top-N.

### Pourquoi la similarité cosinus ?
Parce qu’elle mesure l’angle entre deux vecteurs (orientation), ce qui est robuste aux différences de longueur :
- un CV peut être court ou long,
- une offre peut être plus ou moins détaillée,
mais on compare surtout la **proximité de contenu**.


In [ ]:
from pathlib import Path
from cv.parser import load_all_cvs
from nlp.vectorizer import load_and_prepare_offers, build_tfidf_matrix
from nlp.similarity import rank_offers_for_cv  # adapte si ton nom de fonction est différent

# Chemins
offers_path = Path("data/offer.csv")
cv_dir = Path("data/cv")

# 1) Offres -> TF-IDF
df_offers = load_and_prepare_offers(offers_path)
vectorizer, X_offers_tfidf = build_tfidf_matrix(df_offers["clean_document"].tolist())

# 2) Charger un CV (on prend le premier trouvé)
cvs = load_all_cvs(cv_dir)
cv_name, cv_text = list(cvs.items())[0]

print(f"CV utilisé : {cv_name}")

# 3) Ranking Top-10
top10 = rank_offers_for_cv(
    cv_text=cv_text,
    df_offers=df_offers,
    vectorizer=vectorizer,
    X_offers_tfidf=X_offers_tfidf,
    top_n=10
)

top10[["title", "score"]].head(10)


## 5. Similarité sémantique par LSA — `nlp/lsa.py`

### Objectif
Ce module améliore la comparaison CV ↔ offres en utilisant la **Latent Semantic Analysis (LSA)**.  
Contrairement au TF-IDF brut, la LSA permet de capter des **relations sémantiques globales** entre les mots.

### Principe
1. On part de la matrice TF-IDF des offres.
2. On applique une **décomposition en valeurs singulières (SVD)** afin de réduire la dimension.
3. Chaque offre est projetée dans un espace latent de dimension réduite.
4. Le CV est projeté dans **le même espace latent**.
5. La similarité cosinus est calculée dans cet espace réduit.

### Intérêt par rapport à TF-IDF
- Réduit le bruit lié aux synonymes ou variations lexicales.
- Capture des thématiques communes même si le vocabulaire exact diffère.
- Fournit une vision plus globale et plus robuste du contenu.


In [ ]:
from pathlib import Path
from cv.parser import load_all_cvs
from nlp.vectorizer import load_and_prepare_offers, build_tfidf_matrix
from nlp.lsa import build_lsa_space, rank_offers_lsa  # adapte si besoin

# Chemins
offers_path = Path("data/offer.csv")
cv_dir = Path("data/cv")

# 1) Offres → TF-IDF
df_offers = load_and_prepare_offers(offers_path)
vectorizer, X_offers_tfidf = build_tfidf_matrix(df_offers["clean_document"].tolist())

# 2) Construction de l’espace LSA
svd, X_offers_lsa = build_lsa_space(X_offers_tfidf, n_components=200)

# 3) Charger un CV
cvs = load_all_cvs(cv_dir)
cv_name, cv_text = list(cvs.items())[0]

print(f"CV utilisé : {cv_name}")

# 4) Ranking Top-10 LSA
top10_lsa = rank_offers_lsa(
    cv_text=cv_text,
    df_offers=df_offers,
    vectorizer=vectorizer,
    svd=svd,
    X_offers_lsa=X_offers_lsa,
    top_n=10
)

top10_lsa[["title", "score"]].head(10)


## 6. Traitement multi-CV & export des résultats — `nlp/multi_cv.py`

### Objectif
Ce module automatise l’exécution du matching pour **tous les CV** présents dans `data/cv/` et génère des résultats exploitables.

### Fonctionnement (étapes)
1. Chargement et préparation des offres (construction du document + nettoyage).
2. Entraînement du TF-IDF sur les offres.
3. Construction de l’espace LSA (SVD) à partir du TF-IDF des offres.
4. Chargement de tous les CV PDF du dossier.
5. Pour chaque CV :
   - calcul du Top-10 par TF-IDF,
   - calcul du Top-10 par LSA,
   - calcul de la stabilité via l’overlap (offres communes dans les deux Top-10).
6. Export des résultats dans `data/outputs/` :
   - `<CV>_top_tfidf.csv`
   - `<CV>_top_lsa.csv`
   - `summary.csv` (résumé par CV)

### Résultat attendu
Un système “batch” reproductible qui fournit des fichiers CSV directement utilisables pour la visualisation et le rapport.


In [ ]:
import subprocess
import sys

# Lance le module multi_cv comme en terminal (Windows/Linux)
result = subprocess.run([sys.executable, "-m", "nlp.multi_cv"], capture_output=True, text=True)

print(result.stdout)
if result.stderr:
    print("---- STDERR ----")
    print(result.stderr)


## 7. Orchestration du système — `pipeline.py`

### Objectif
`pipeline.py` centralise l’exécution du projet en une seule fonction (souvent `run_pipeline`).  
Il orchestre toutes les étapes : préparation des offres, apprentissage des modèles, matching des CV, et export des résultats.

### Fonctionnement (étapes)
1. **Chargement des offres** depuis `data/offer.csv`.
2. **Préparation des offres** : construction d’un texte complet par annonce + nettoyage.
3. **Apprentissage du TF-IDF** sur toutes les offres (création du vocabulaire + matrice TF-IDF).
4. **Construction de l’espace LSA** via SVD à partir du TF-IDF.
5. **Chargement de tous les CV** depuis `data/cv/`.
6. Pour chaque CV :
   - ranking Top-N TF-IDF,
   - ranking Top-N LSA,
   - comparaison des classements (overlap).
7. **Export** des fichiers CSV dans `data/outputs/`.

### Pourquoi c’est important
Ce fichier fournit une API claire et stable : un seul appel permet de produire tous les résultats,
ce qui facilite l’intégration, la reproductibilité et la présentation.


In [ ]:
from pathlib import Path
from pipeline import run_pipeline

offers_csv = Path("data/offer.csv")
cv_dir = Path("data/cv")
out_dir = Path("data/outputs")

run_pipeline(
    offers_csv=offers_csv,
    cv_dir=cv_dir,
    out_dir=out_dir,
    top_n=10,
    n_components=200
)


## 8. Point d’entrée du projet — `main.py`

### Objectif
`main.py` est le **point d’entrée unique** du projet.  
Il permet de lancer l’ensemble du pipeline sans avoir à manipuler les modules internes.

### Rôle dans l’architecture
- Il définit les **chemins des données** (CV, offres, outputs).
- Il appelle la fonction centrale `run_pipeline` définie dans `pipeline.py`.
- Il ne contient **aucune logique métier** : toute l’intelligence est déléguée au pipeline.

### Pourquoi séparer `main.py` et `pipeline.py` ?
- `pipeline.py` : logique réutilisable (appelable depuis un notebook, un test, une API).
- `main.py` : exécution simple en ligne de commande (`python main.py`).

Cette séparation facilite l’intégration, les tests et l’évolution du projet.


In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "main.py"],
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print("---- STDERR ----")
    print(result.stderr)
